# Dataset Integrity Check

End-to-end verification: instantiate each PyTorch Dataset via `get_datasets()`,
iterate through **every sample**, and collect statistics on shapes, values, and errors.

In [1]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import time
import traceback
from dataclasses import dataclass, field

import torch
from torch.utils.data import DataLoader, Dataset
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from radharmony.dataset import (
    BRAXDataset,
    BRAXPNGDataset,
    ChestXray14TrainDataset,
    ChestXray14TestDataset,
    ChestXray14BboxDataset,
    CheXpertTrainDataset,
    CheXpertValidDataset,
    CheXpertPlusDataset,
    MIMICCXRDataset,
    MIMICCXRJPGDataset,
    MIMICCXRJPGTestDataset,
    OpenICXRDataset,
    PadChestDataset,
    ReXGradientTrainDataset,
    ReXGradientValidDataset,
    ReXGradientTestDataset,
    RANZCRClipDataset,
    VinDrCXRTrainDataset,
    VinDrCXRTestDataset,
    SIIMACRPTXTrainDataset,
    SIIMCOVID19TrainDataset,
    RSNAPneumoniaKaggleTrainDataset,
    MontgomeryCXRDataset,
    ShenzhenCXRDataset,
    TAIXRay512Dataset,
    TAIXRayDataset,
    RSNAPEDetectionTrainDataset,
    RSNABoneAgeTrainDataset,
    RSNABoneAgeValDataset,
    RSNA2022CervicalSpineTrainDataset,
    RSNA2022CervicalSpineBboxDataset,
    RSNA2024LumbarSpineTrainDataset,
    RSNAAbdominalTrauma2023Dataset,
    CTRATEDataset,
    RadChestCTDataset,
)

In [5]:
# Optionally pin a GPU (dataset loading itself is CPU-bound):
# import os
# os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [6]:
@dataclass
class IntegrityReport:
    """Statistics from iterating every sample in a dataset."""

    dataset: str = ""
    status: str = "OK"  # OK | WARN | ERROR
    error_message: str = ""
    total_time_s: float = 0.0

    # Counts
    total_samples: int = 0
    success: int = 0
    failed: int = 0
    failed_indices: list = field(default_factory=list)  # first N failed
    failed_errors: list = field(default_factory=list)  # corresponding error msgs
    failed_filenames: list = field(default_factory=list)  # corresponding file names

    # Image stats (from successful samples)
    img_shapes: dict = field(default_factory=dict)  # shape_str -> count
    img_dtype: str = ""
    img_min: float = float("inf")
    img_max: float = float("-inf")
    img_nan_count: int = 0
    img_inf_count: int = 0

    # Cls stats
    has_cls: bool = False
    cls_length: int = 0
    cls_all_nan_count: int = 0  # samples where every label is nan
    cls_pos_counts: dict = field(default_factory=dict)  # label_idx -> positive count

    # Mask stats
    has_mask: bool = False
    mask_shapes: dict = field(default_factory=dict)
    mask_empty_count: int = 0  # masks that are all zeros

    # Report stats
    has_report: bool = False
    report_count: int = 0
    report_empty_count: int = 0

    # BBox stats
    has_bbox: bool = False
    bbox_count: int = 0
    bbox_invalid_count: int = 0  # values outside [0, 1] or wrong length

    # Keys present
    sample_keys: list = field(default_factory=list)

    # Live reference to the dataset wrapper used in the integrity pass.  Kept
    # so the bottom of the notebook can call ``ds_obj.visualize_samples(...)``
    # without re-instantiating the wrapper.
    ds_obj: object = None

    def summary_dict(self) -> dict:
        shapes = ", ".join(f"{s}: {c}" for s, c in sorted(self.img_shapes.items()))
        return {
            "dataset": self.dataset,
            "status": self.status,
            "samples": self.total_samples,
            "success": self.success,
            "failed": self.failed,
            "img_range": (
                f"[{self.img_min:.3f}, {self.img_max:.3f}]" if self.success else "-"
            ),
            "img_nan": self.img_nan_count,
            "img_shapes": shapes,
            "time_s": f"{self.total_time_s:.1f}",
        }

In [7]:
class _SafeDataset(Dataset):
    """Wraps a dataset so that per-sample errors are returned as (None, idx, err_str)
    instead of raising, allowing DataLoader workers to keep going."""

    def __init__(self, dataset):
        self.dataset = dataset

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, i):
        try:
            return self.dataset[i], i, ""
        except Exception as e:
            return None, i, f"{type(e).__name__}: {e}"


def _collate(batch):
    """Collate a list of (sample_or_None, idx, err) tuples into a batch dict."""
    samples, indices, errors = zip(*batch)
    out = {"_idx": list(indices), "_err": list(errors)}
    # only collate successful samples
    good = [s for s in samples if s is not None]
    if not good:
        return out

    for key in good[0].keys():
        vals = [s[key] for s in good]
        if isinstance(vals[0], torch.Tensor):
            try:
                out[key] = torch.stack(vals)
            except Exception:
                out[key] = vals  # variable shapes — fall back to list
        else:
            out[key] = vals
    return out


def check_dataset(
    ds_obj,
    label_cols: list = None,
    max_errors: int = 20,
    num_workers: int = 8,
    batch_size: int = 8,
) -> "IntegrityReport":
    """Instantiate a dataset via get_datasets() and iterate every sample with a DataLoader.

    Args:
        ds_obj: An instantiated BaseRadiologicalDataset subclass.
        label_cols: Label column names for cls reporting. Defaults to ds_obj.LABEL_COLS.
        max_errors: Max number of failed sample details to keep.
        num_workers: DataLoader worker processes (0 = main process only).
        batch_size: Samples per batch.

    Returns:
        IntegrityReport with per-sample statistics.  ``report.ds_obj`` holds
        the dataset wrapper so plotting can call ``visualize_samples``
        without re-building it.
    """
    name = type(ds_obj).__name__
    report = IntegrityReport(dataset=name, ds_obj=ds_obj)
    label_cols = label_cols or getattr(ds_obj, "LABEL_COLS", [])

    # --- Build the dataset ---
    try:
        t0 = time.time()
        dataset = ds_obj.get_datasets()
        build_time = time.time() - t0
    except Exception as e:
        report.status = "ERROR"
        report.error_message = f"get_datasets() failed: {e}"
        return report

    report.total_samples = len(dataset)
    if report.total_samples == 0:
        report.status = "WARN"
        report.error_message = "Dataset has 0 samples."
        return report

    # --- DataLoader ---
    loader = DataLoader(
        _SafeDataset(dataset),
        batch_size=batch_size,
        num_workers=num_workers,
        shuffle=False,
        collate_fn=_collate,
        prefetch_factor=2 if num_workers > 0 else None,
        # persistent_workers=num_workers > 0,
        pin_memory=True,
    )

    first_good_seen = False
    t0 = time.time()

    for batch in tqdm(loader, desc=name):
        # ── Track errors from this batch ──
        for idx, err in zip(batch["_idx"], batch["_err"]):
            if err:
                report.failed += 1
                if len(report.failed_indices) < max_errors:
                    report.failed_indices.append(int(idx))
                    report.failed_errors.append(err)
                    try:
                        fname = str(dataset.data[int(idx)]["img"])
                    except Exception:
                        fname = "N/A"
                    report.failed_filenames.append(fname)

        if "img" not in batch:
            continue  # entire mini-batch failed

        n_good = (
            batch["img"].shape[0]
            if isinstance(batch["img"], torch.Tensor)
            else len(batch["img"])
        )
        report.success += n_good

        # ── First-sample metadata ──
        if not first_good_seen:
            first_good_seen = True
            report.sample_keys = sorted(k for k in batch if not k.startswith("_"))
            report.has_cls = "cls" in batch
            report.has_mask = "mask" in batch
            report.has_report = "report" in batch
            report.has_bbox = "bbox" in batch
            if report.has_cls:
                cls0 = batch["cls"]
                if isinstance(cls0, torch.Tensor):
                    report.cls_length = cls0.shape[-1]
                elif hasattr(cls0[0], "__len__"):
                    report.cls_length = len(cls0[0])
                else:
                    report.cls_length = 1

        # ── img ──
        img = batch["img"]
        if isinstance(img, torch.Tensor):
            # img shape: (B, C, ...) — per-sample shape is img.shape[1:]
            shape_str = str(tuple(img.shape[1:]))
            report.img_shapes[shape_str] = report.img_shapes.get(shape_str, 0) + n_good
            if not report.img_dtype:
                report.img_dtype = str(img.dtype)
            img_f = img.float()
            report.img_min = min(report.img_min, img_f.min().item())
            report.img_max = max(report.img_max, img_f.max().item())
            if torch.isnan(img_f).any():
                report.img_nan_count += (
                    torch.isnan(img_f).flatten(1).any(1).sum().item()
                )
            if torch.isinf(img_f).any():
                report.img_inf_count += (
                    torch.isinf(img_f).flatten(1).any(1).sum().item()
                )
        else:  # list of tensors (variable shapes)
            for t in img:
                shape_str = str(tuple(t.shape))
                report.img_shapes[shape_str] = report.img_shapes.get(shape_str, 0) + 1
                if not report.img_dtype:
                    report.img_dtype = str(t.dtype)
                tf = t.float()
                report.img_min = min(report.img_min, tf.min().item())
                report.img_max = max(report.img_max, tf.max().item())
                if torch.isnan(tf).any():
                    report.img_nan_count += 1
                if torch.isinf(tf).any():
                    report.img_inf_count += 1

        # ── cls ──
        if report.has_cls:
            cls_batch = batch["cls"]
            if isinstance(cls_batch, torch.Tensor):
                cls_f = cls_batch.float()  # (B, L)
                for row in cls_f:
                    vals = row.tolist()
                    if all(np.isnan(v) for v in vals):
                        report.cls_all_nan_count += 1
                    for idx, v in enumerate(vals):
                        if not np.isnan(v) and v > 0.5:
                            report.cls_pos_counts[idx] = (
                                report.cls_pos_counts.get(idx, 0) + 1
                            )
            else:
                for item in cls_batch:
                    vals = (
                        item.float().tolist()
                        if hasattr(item, "float")
                        else [float(v) for v in item]
                    )
                    if all(np.isnan(v) for v in vals):
                        report.cls_all_nan_count += 1
                    for idx, v in enumerate(vals):
                        if not np.isnan(v) and v > 0.5:
                            report.cls_pos_counts[idx] = (
                                report.cls_pos_counts.get(idx, 0) + 1
                            )

        # ── mask ──
        if report.has_mask:
            masks = batch["mask"]
            items = (
                masks
                if isinstance(masks, list)
                else [masks[i] for i in range(masks.shape[0])]
            )
            for m in items:
                ms = str(tuple(m.shape))
                report.mask_shapes[ms] = report.mask_shapes.get(ms, 0) + 1
                if m.float().sum().item() == 0:
                    report.mask_empty_count += 1

        # ── report ──
        if report.has_report:
            for rpt in batch["report"]:
                report.report_count += 1
                if not str(rpt).strip():
                    report.report_empty_count += 1

        # ── bbox ──
        if report.has_bbox:
            for coords in batch["bbox"]:
                report.bbox_count += 1
                try:
                    if hasattr(coords, "tolist"):
                        coords = coords.tolist()
                    if len(coords) not in (4, 6) or not all(
                        0.0 <= float(c) <= 1.0 for c in coords
                    ):
                        report.bbox_invalid_count += 1
                except Exception:
                    report.bbox_invalid_count += 1

    report.total_time_s = (time.time() - t0) + build_time

    if report.failed > 0:
        report.status = "WARN" if report.success > 0 else "ERROR"

    return report


def print_report(r: "IntegrityReport", label_cols: list = None) -> None:
    """Pretty-print a single integrity report."""
    icon = {"OK": "✅", "WARN": "⚠️", "ERROR": "❌"}
    print(f"\n{'='*60}")
    print(f"{icon.get(r.status, '?')}  {r.dataset}  [{r.status}]")
    print(f"{'='*60}")

    if r.error_message:
        print(f"  Error: {r.error_message}")
        if r.status == "ERROR" and r.success == 0:
            return

    print(f"  Total time     : {r.total_time_s:.1f}s")
    print(f"  Samples        : {r.total_samples:,}")
    print(f"  Success        : {r.success:,}")
    print(f"  Failed         : {r.failed:,}")
    print(f"  Sample keys    : {r.sample_keys}")
    print()

    print(f"  img dtype      : {r.img_dtype}")
    print(f"  img range      : [{r.img_min:.4f}, {r.img_max:.4f}]")
    print(f"  img NaN        : {r.img_nan_count:,} samples")
    print(f"  img Inf        : {r.img_inf_count:,} samples")
    print(f"  img shapes     :")
    for s, c in sorted(r.img_shapes.items(), key=lambda x: -x[1]):
        print(f"    {s:30s} x {c:,}")

    if r.has_cls:
        print(f"\n  cls vector len : {r.cls_length}")
        print(f"  cls all-NaN    : {r.cls_all_nan_count:,} samples")
        label_cols = label_cols or []
        print(f"  cls positive counts:")
        for idx in range(r.cls_length):
            name = label_cols[idx] if idx < len(label_cols) else f"[{idx}]"
            pos = r.cls_pos_counts.get(idx, 0)
            pct = pos / r.success * 100 if r.success else 0
            print(f"    {name:40s}  {pos:>6,} / {r.success:,} ({pct:5.1f}%)")

    if r.has_mask:
        print(f"\n  mask shapes    :")
        for s, c in sorted(r.mask_shapes.items(), key=lambda x: -x[1]):
            print(f"    {s:30s} x {c:,}")
        print(f"  mask all-zero  : {r.mask_empty_count:,} / {r.success:,}")

    if r.has_report:
        print(f"\n  reports        : {r.report_count:,}")
        print(f"  reports empty  : {r.report_empty_count:,}")

    if r.has_bbox:
        print(f"\n  bboxes         : {r.bbox_count:,}")
        print(f"  bboxes invalid : {r.bbox_invalid_count:,}")

    if r.failed_indices:
        print(f"\n  First {len(r.failed_indices)} failures:")
        for idx, err, fname in zip(
            r.failed_indices, r.failed_errors, r.failed_filenames
        ):
            print(f"    [{idx}] {fname}")
            print(f"           {err}")

## Configure dataset paths

Edit the paths below. Comment out any dataset you don't have locally.

In [ ]:
# ── Dataset paths (local NAS) ──────────────────────────────────────────
# Edit for your environment. Comment out any dataset you don'''t have.

# CXR (2-D)
CHEXPERT_TRAIN_BASE   = "/mnt/NAS3/datasets/external/CheXpert-v1.0/train/"
CHEXPERT_VALID_BASE   = "/mnt/NAS3/datasets/external/CheXpert-v1.0/valid/"
CHEXPERT_PLUS_BASE    = "/mnt/NAS4/datasets/external/CheXpert_Plus/chexpertplus/DICOM/Uncompressed/"

# MIMIC-CXR metadata + label CSVs ship in the JPG release; pass them
# explicitly if they live apart from the image tree.
MIMIC_CXR_BASE        = "/mnt/NAS4/datasets/external/MIMIC-CXR-V2-AWS/files/"
MIMIC_CXR_JPG_BASE    = "/mnt/NAS3/datasets/external/MIMIC_CXR/physionet.org/files/mimic-cxr-jpg/2.0.0/files/"
MIMIC_META_CSV        = "/mnt/NAS3/datasets/external/MIMIC_CXR/physionet.org/files/mimic-cxr-jpg/2.0.0/mimic-cxr-2.0.0-metadata.csv"
MIMIC_CHEXPERT_CSV    = "/mnt/NAS3/datasets/external/MIMIC_CXR/physionet.org/files/mimic-cxr-jpg/2.0.0/mimic-cxr-2.0.0-chexpert.csv"

CHESTXRAY14_BASE      = "/mnt/NAS3/datasets/external/NIH_CXR/CXR14/"
PADCHEST_BASE         = "/mnt/NAS4/datasets/external/PadChest/images/"
BRAX_BASE             = "/mnt/NAS4/datasets/external/BRAX/"
REXGRADIENT_BASE      = "/mnt/NAS4/datasets/external/ReXGradient-160K/deid_png/"
OPENI_BASE            = "/mnt/NAS4/datasets/external/OpenI-IU-CXR/"
RANZCR_CLIP_BASE      = "/mnt/NAS4/datasets/external/ranzcr-clip-catheter-line-classification/"
VINDR_CXR_TRAIN_BASE  = "/mnt/NAS4/datasets/external/VinDr-CXR/physionet.org/files/vindr-cxr/1.0.0/train/"
VINDR_CXR_TEST_BASE   = "/mnt/NAS4/datasets/external/VinDr-CXR/physionet.org/files/vindr-cxr/1.0.0/test/"
SIIM_ACR_BASE         = "/mnt/NAS3/datasets/external/SIIM_ACR_Pneumothorax/dicom-images-train/"
SIIM_COVID19_BASE     = "/mnt/NAS4/datasets/external/siim-covid19-detection/train/"
RSNA_PNEUMONIA_BASE   = "/mnt/NAS4/datasets/external/rsna-pneumonia-detection-challenge/"
TAIX_RAY_512_BASE     = "/mnt/NAS4/datasets/external/TAIX-Ray/data_512/images/"
MONTGOMERY_CXR_BASE   = "/mnt/NAS4/datasets/external/Montgomery-CXR/MontgomerySet/CXR_png/"
SHENZHEN_CXR_BASE     = "/mnt/NAS4/datasets/external/Shenzhen-Hospital-CXR-Set/CXR_png/"

# CT / MRI (3-D)
RSNA_PE_DETECTION_BASE   = "/mnt/NAS4/datasets/external/rsna_pe_dataset/train/"
RSNA_BONE_AGE_TRAIN_BASE = "/mnt/NAS4/datasets/external/rsna-boneage/boneage-training-dataset/"
RSNA_2022_CERVICAL_BASE  = "/mnt/NAS4/datasets/external/rsna-2022-cervical-spine-fracture-detection/train_images/"
RSNA_2023_ABDOMINAL_BASE = "/mnt/NAS4/datasets/external/rsna-2023-abdominal-trauma-detection/train_images/"
RSNA_2024_LUMBAR_BASE    = "/mnt/NAS4/datasets/external/rsna-2024-lumbar-spine-degenerative-classification/train_images/"
CTRATE_BASE              = "/mnt/NAS4/datasets/external/CT-RATE/dataset/train_fixed/"
RADCHESTCT_BASE          = "/mnt/NAS4/datasets/external/RAD-ChestCT/images/"

# Scratch dir for datasets that rasterize/decode masks to PNG on the fly.
MASK_OUT = os.path.expanduser("~/.cache/radharmony/verify_masks")
os.makedirs(MASK_OUT, exist_ok=True)

## Verification driver

`build(DatasetClass, **kwargs)` does the whole loop for one dataset:

1. **instantiate** the dataset with the given (all supported) arguments — this
   runs the harmonizer end-to-end;
2. **sub-sample** the harmonized table to `MAX_SAMPLES` rows for a fast pass
   (set `MAX_SAMPLES = None` for a full integrity sweep of every sample);
3. **iterate** every (sub-sampled) sample through a `DataLoader` and collect
   shape / range / NaN / error statistics (`check_dataset`);
4. **preview** — loop a few samples, **print the decoded labels / report /
   bbox**, and **visualize** the image (2-D directly, 3-D as an axial mean
   projection; masks and bboxes overlaid when present).

Per-sample and per-dataset errors are captured, not raised, so one broken
dataset never aborts the run — it just shows up as `ERROR`/`WARN` in its report
and in the final summary table.

In [ ]:
# ── Verification driver ───────────────────────────────────────────────────
MAX_SAMPLES = 60    # rows to sub-sample per dataset (None = every sample)
N_PREVIEW   = 4     # samples to print + visualize per dataset


def _subsample(ds):
    """Pin a small random subset of the harmonized table for a fast pass."""
    df = ds.get_harmonized_df()
    n = len(df)
    if MAX_SAMPLES and n > MAX_SAMPLES:
        ds.set_harmonized_df(
            df.sample(n=MAX_SAMPLES, random_state=42).reset_index(drop=True)
        )
        print(f"  sub-sampled {n:,} -> {MAX_SAMPLES} rows (set MAX_SAMPLES=None for all)")


def preview(r, n=N_PREVIEW):
    """Loop a few samples through a DataLoader: print labels/report/bbox, show figures."""
    import matplotlib.pyplot as plt

    ds_obj = r.ds_obj
    if ds_obj is None:
        print("  (dataset failed to build - nothing to preview)")
        return
    dataset = ds_obj.get_datasets()
    lcols = sorted(ds_obj.LABEL_COLS) if getattr(ds_obj, "LABEL_COLS", None) else []
    rcols = sorted(getattr(ds_obj, "REG_COLS", []) or [])

    loader = DataLoader(
        _SafeDataset(dataset), batch_size=1, num_workers=0, collate_fn=_collate
    )
    print(f"  -- looping {min(n, len(dataset))} sample(s) through a DataLoader --")
    shown = 0
    for batch in loader:
        idx = batch["_idx"][0]
        if "img" not in batch:
            print(f"    sample[{idx}] FAILED: {batch['_err'][0]}")
            continue
        img = batch["img"][0]
        print(f"    sample[{idx}]  img shape={tuple(img.shape)} dtype={img.dtype}")
        if "cls" in batch:
            v = batch["cls"][0]
            v = v.float().tolist() if hasattr(v, "float") else [float(x) for x in v]
            pos = [lcols[j] for j, x in enumerate(v)
                   if j < len(lcols) and not np.isnan(x) and x > 0.5]
            print(f"        cls[{len(v)}] positive: {pos if pos else '(none)'}")
        if "reg" in batch:
            v = batch["reg"][0]
            v = v.float().tolist() if hasattr(v, "float") else list(v)
            print(f"        reg {rcols} = {v}")
        if "bbox" in batch:
            labs = batch.get("bbox_labels", [None])[0]
            print(f"        bbox: {len(batch['bbox'][0])} box(es)  labels={labs}")
        if "report" in batch:
            print(f"        report: {str(batch['report'][0])[:150].strip()} ...")
        if "previous_img" in batch:
            print(f"        previous_img shape={tuple(batch['previous_img'][0].shape)}")
        shown += 1
        if shown >= n:
            break

    try:
        for fig in ds_obj.visualize_samples(dataset, n=n, random_state=0):
            plt.show()
            plt.close(fig)
    except Exception as e:
        print(f"  [warn] visualize_samples failed: {type(e).__name__}: {e}")


def build(ds_cls, *, num_workers=4, **kwargs):
    """Instantiate -> sub-sample -> integrity-check -> preview one dataset."""
    kwargs.setdefault("cache_dir", None)
    shown = {k: v for k, v in kwargs.items() if k != "cache_dir"}
    print("\n" + "#" * 72)
    print(f"# {ds_cls.__name__}")
    for k, v in shown.items():
        print(f"#   {k} = {v}")
    print("#" * 72)
    try:
        ds = ds_cls(**kwargs)
    except Exception as e:
        traceback.print_exc()
        r = IntegrityReport(dataset=ds_cls.__name__, status="ERROR",
                            error_message=f"__init__ failed: {e}")
        reports.append((r, []))
        return r
    try:
        _subsample(ds)
    except Exception as e:
        print(f"  [warn] could not sub-sample: {type(e).__name__}: {e}")
    r = check_dataset(ds, num_workers=num_workers)
    print_report(r, ds.LABEL_COLS)
    preview(r)
    reports.append((r, ds.LABEL_COLS))
    return r

## Run verification

Each cell builds one dataset, checks integrity over a sub-sample, prints labels, and shows previews. Run top-to-bottom or cherry-pick the datasets you have.

In [ ]:
reports = []  # (IntegrityReport, label_cols) per dataset

### Chest X-ray (2-D)

In [ ]:
# CheXpert — train (14 pathologies)
build(CheXpertTrainDataset, base_image_dir=CHEXPERT_TRAIN_BASE, output_cls=True)

In [ ]:
# CheXpert — valid
build(CheXpertValidDataset, base_image_dir=CHEXPERT_VALID_BASE, output_cls=True)

In [ ]:
# CheXpert-Plus (DICOM) — cls + de-identified reports
build(CheXpertPlusDataset, base_image_dir=CHEXPERT_PLUS_BASE,
      output_cls=True, output_report=True, drop_uncertain=False)

In [ ]:
# MIMIC-CXR (DICOM) — cls + reports (metadata/labels may live apart from the image tree)
build(MIMICCXRDataset, base_image_dir=MIMIC_CXR_BASE,
      csv_path=MIMIC_META_CSV, label_csv_path=MIMIC_CHEXPERT_CSV,
      output_cls=True, output_report=True, drop_uncertain=False)

In [ ]:
# MIMIC-CXR-JPG — cls (JPG release ships no reports)
build(MIMICCXRJPGDataset, base_image_dir=MIMIC_CXR_JPG_BASE,
      csv_path=MIMIC_META_CSV, label_csv_path=MIMIC_CHEXPERT_CSV, output_cls=True)

In [ ]:
# MIMIC-CXR-JPG — labeled test set
build(MIMICCXRJPGTestDataset, base_image_dir=MIMIC_CXR_JPG_BASE,
      csv_path=MIMIC_META_CSV, output_cls=True)

In [ ]:
# ChestX-ray14 — train (15 classes)
build(ChestXray14TrainDataset, base_image_dir=CHESTXRAY14_BASE, output_cls=True)

In [ ]:
# ChestX-ray14 — test
build(ChestXray14TestDataset, base_image_dir=CHESTXRAY14_BASE, output_cls=True)

In [ ]:
# ChestX-ray14 — bbox subset (cls + bbox)
build(ChestXray14BboxDataset, base_image_dir=CHESTXRAY14_BASE,
      output_cls=True, output_bbox=True)

In [ ]:
# PadChest — 193 labels + Spanish reports
build(PadChestDataset, base_image_dir=PADCHEST_BASE,
      output_cls=True, output_report=True)

In [ ]:
# BRAX (DICOM) — cls (uncertain -1 -> 0)
build(BRAXDataset, base_image_dir=BRAX_BASE,
      uncertain_strategy="u_zeros", output_cls=True)

In [ ]:
# BRAX (PNG) — cls
build(BRAXPNGDataset, base_image_dir=BRAX_BASE,
      uncertain_strategy="u_zeros", output_cls=True)

In [ ]:
# OpenI IU CXR — cls (CheXpert-compat) + reports
build(OpenICXRDataset, base_image_dir=OPENI_BASE,
      output_cls=True, output_report=True)

In [ ]:
# ReXGradient-160K — valid (report only, no labels)
build(ReXGradientValidDataset, base_image_dir=REXGRADIENT_BASE, output_report=True)

In [ ]:
# ReXGradient-160K — test
build(ReXGradientTestDataset, base_image_dir=REXGRADIENT_BASE, output_report=True)

In [ ]:
# ReXGradient-160K — train (large; sub-sampled after harmonize)
build(ReXGradientTrainDataset, base_image_dir=REXGRADIENT_BASE, output_report=True)

In [ ]:
# VinDr-CXR — train (cls + bbox)
build(VinDrCXRTrainDataset, base_image_dir=VINDR_CXR_TRAIN_BASE,
      output_cls=True, output_bbox=True)

In [ ]:
# VinDr-CXR — test
build(VinDrCXRTestDataset, base_image_dir=VINDR_CXR_TEST_BASE,
      output_cls=True, output_bbox=True)

In [ ]:
# SIIM-ACR Pneumothorax — cls + mask (RLE decoded to PNG under MASK_OUT)
build(SIIMACRPTXTrainDataset, base_image_dir=SIIM_ACR_BASE,
      mask_output_dir=os.path.join(MASK_OUT, "siim_acr"),
      output_cls=True, output_mask=True)

In [ ]:
# SIIM COVID-19 — cls (4 study classes) + bbox
build(SIIMCOVID19TrainDataset, base_image_dir=SIIM_COVID19_BASE,
      output_cls=True, output_bbox=True)

In [ ]:
# RSNA Pneumonia (Kaggle) — cls (3 classes) + bbox
build(RSNAPneumoniaKaggleTrainDataset, base_image_dir=RSNA_PNEUMONIA_BASE,
      output_cls=True, output_bbox=True)

In [ ]:
# RANZCR CLiP — cls (11 line/catheter labels). Enable mask via the next cell.
build(RANZCRClipDataset, base_image_dir=RANZCR_CLIP_BASE,
      include_test_split=False, output_cls=True)

In [ ]:
# RANZCR CLiP — cls + rasterized polyline masks (slower: masks are
# generated at harmonize time). Restrict to annotated rows so every
# sample has a real mask.
build(RANZCRClipDataset, base_image_dir=RANZCR_CLIP_BASE,
      mask_output_dir=os.path.join(MASK_OUT, "ranzcr"),
      include_test_split=False, output_cls=True, output_mask=True)

In [ ]:
# TAIX-Ray 512px — cls, ordinal severity grades (0-4)
build(TAIXRay512Dataset, base_image_dir=TAIX_RAY_512_BASE,
      label_mode="ordinal", output_cls=True)

In [ ]:
# Montgomery County CXR — TB cls + lung mask + report
build(MontgomeryCXRDataset, base_image_dir=MONTGOMERY_CXR_BASE,
      mask_output_dir=os.path.join(MASK_OUT, "montgomery"),
      output_cls=True, output_mask=True, output_report=True)

In [ ]:
# Shenzhen Hospital CXR — TB cls + mask + report
build(ShenzhenCXRDataset, base_image_dir=SHENZHEN_CXR_BASE,
      mask_output_dir=os.path.join(MASK_OUT, "shenzhen"),
      output_cls=True, output_mask=True, output_report=True)

### CT / MRI (3-D)

In [ ]:
# RSNA PE Detection (3-D CTPA series) — cls
build(RSNAPEDetectionTrainDataset, base_image_dir=RSNA_PE_DETECTION_BASE,
      output_cls=True, num_workers=2)

In [ ]:
# RSNA Pediatric Bone Age (2-D) — regression (age_months)
build(RSNABoneAgeTrainDataset, base_image_dir=RSNA_BONE_AGE_TRAIN_BASE, output_reg=True)

In [ ]:
# RSNA 2022 Cervical Spine (3-D) — cls + NIfTI seg mask
build(RSNA2022CervicalSpineTrainDataset, base_image_dir=RSNA_2022_CERVICAL_BASE,
      output_cls=True, output_mask=True, num_workers=2)

In [ ]:
# RSNA 2022 Cervical Spine — bbox variant (cls + 3-D bbox)
build(RSNA2022CervicalSpineBboxDataset, base_image_dir=RSNA_2022_CERVICAL_BASE,
      output_cls=True, output_bbox=True, num_workers=2)

In [ ]:
# RSNA 2024 Lumbar Spine (3-D MRI) — cls (75 cols) + bbox
build(RSNA2024LumbarSpineTrainDataset, base_image_dir=RSNA_2024_LUMBAR_BASE,
      output_cls=True, output_bbox=True, num_workers=2)

In [ ]:
# RSNA 2023 Abdominal Trauma (3-D) — cls (14 injury labels)
build(RSNAAbdominalTrauma2023Dataset, base_image_dir=RSNA_2023_ABDOMINAL_BASE,
      output_cls=True, num_workers=2)

In [ ]:
# CT-RATE (3-D NIfTI) — cls (18 findings)
build(CTRATEDataset, base_image_dir=CTRATE_BASE, output_cls=True, num_workers=2)

In [ ]:
# RAD-ChestCT (3-D .npz) — cls (83/84 findings) + bbox
build(RadChestCTDataset, base_image_dir=RADCHESTCT_BASE,
      output_cls=True, output_bbox=True, num_workers=2)

## Summary table

In [24]:
summary_df = pd.DataFrame([r.summary_dict() for r, _ in reports])
summary_df

,dataset,status,samples,success,failed,img_range,img_nan,img_shapes,time_s
0,ChestXray14Dataset,OK,110726,110726,0,"[-1.000, 1.000]",0,"(1, 224, 224): 110726",412.5
1,ChestXray14BboxDataset,OK,875,875,0,"[-1.000, 1.000]",0,"(1, 224, 224): 875",4.5
2,CheXpertDataset,OK,118286,118286,0,"[-256.000, 1.000]",0,"(1, 224, 224): 118286",1086.3
3,CheXpertPlusDataset,OK,210969,210969,0,"[-474.000, 1.000]",0,"(1, 224, 224): 210969",3623.4
4,MIMICCXRDataset,OK,377094,377094,0,"[-1.000, 1.000]",0,"(1, 224, 224): 377094",10389.5
5,MIMICCXRJPGDataset,OK,297116,297116,0,"[-1.000, 1.000]",0,"(1, 224, 224): 297116",4911.7
6,SIIMACRPTXDataset,OK,11582,11582,0,"[-1.000, 1.000]",0,"(1, 224, 224): 11582",49.4
7,SIIMCOVID19Dataset,OK,6334,6334,0,"[-1.000, 1.000]",0,"(1, 224, 224): 6334",295.9
8,RSNAPneumoniaKaggleDataset,OK,30227,30227,0,"[-1.000, 1.000]",0,"(1, 224, 224): 30227",98.6


## Label distribution overview

In [25]:
for r, lcols in reports:
    if r.status == "ERROR" or not r.has_cls:
        continue
    rows = []
    for idx in range(r.cls_length):
        name = lcols[idx] if idx < len(lcols) else f"[{idx}]"
        pos = r.cls_pos_counts.get(idx, 0)
        neg = r.success - pos
        rows.append(
            {
                "label": name,
                "positive": pos,
                "negative": neg,
                "pos_rate": f"{pos / r.success * 100:.1f}%" if r.success else "-",
            }
        )
    print(f"\n--- {r.dataset} ---")
    display(pd.DataFrame(rows))


--- ChestXray14Dataset ---


,label,positive,negative,pos_rate
0,atelectasis,11220,99506,10.1%
1,cardiomegaly,2587,108139,2.3%
2,consolidation,4581,106145,4.1%
3,edema,2253,108473,2.0%
4,effusion,12959,97767,11.7%
5,emphysema,2473,108253,2.2%
6,fibrosis,1658,109068,1.5%
7,hernia,225,110501,0.2%
8,infiltration,19523,91203,17.6%
9,mass,5628,105098,5.1%



--- ChestXray14BboxDataset ---


,label,positive,negative,pos_rate
0,atelectasis,302,573,34.5%
1,cardiomegaly,182,693,20.8%
2,consolidation,73,802,8.3%
3,edema,40,835,4.6%
4,effusion,312,563,35.7%
5,emphysema,34,841,3.9%
6,fibrosis,18,857,2.1%
7,hernia,2,873,0.2%
8,infiltration,312,563,35.7%
9,mass,137,738,15.7%



--- CheXpertDataset ---


...

,label,positive,negative,pos_rate
0,pneumothorax,3286,8296,28.4%



--- SIIMCOVID19Dataset ---


,label,positive,negative,pos_rate
0,atypical_appearance,483,5851,7.6%
1,indeterminate_appearance,1108,5226,17.5%
2,negative_for_pneumonia,1736,4598,27.4%
3,typical_appearance,3007,3327,47.5%



--- RSNAPneumoniaKaggleDataset ---


,label,positive,negative,pos_rate
0,lung_opacity,9555,20672,31.6%
1,no_lung_opacity_/_not_normal,11821,18406,39.1%
2,normal,8851,21376,29.3%
